<a href="https://colab.research.google.com/github/mshinno26/UnderstandingAI/blob/main/parkinsons_knn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Dataset Explanation: Parkinson's Disease Classification

This dataset contains various biomedical voice measurements from individuals, used to predict the presence or absence of Parkinson's disease.

**Features (Input, `X`):**
The `X` dataframe contains a set of voice-related features, which are measurements extracted from speech signals. These features are numerical and describe different aspects of voice acoustics. Some key features include:
*   **MDVP:Fo, MDVP:Fhi, MDVP:Flo**: Average, maximum, and minimum vocal fundamental frequency.
*   **MDVP:Jitter, MDVP:RAP, MDVP:PPQ, Jitter:DDP**: Measures of variations in fundamental frequency, often associated with voice instability.
*   **MDVP:Shimmer, Shimmer:APQ, MDVP:APQ, Shimmer:DDA**: Measures of variations in amplitude, also related to voice instability.
*   **NHR, HNR**: Noise-to-harmonics ratio and harmonics-to-noise ratio, indicating the level of noise in the voice.
*   **RPDE, DFA**: Recurrence period density entropy and detrended fluctuation analysis, measures of non-linear dynamics.
*   **spread1, spread2, D2, PPE**: Various measures of vocal fold vibration and dysphonia (voice impairment).

These features collectively provide a comprehensive acoustic profile of an individual's voice.

**Target (Output, `y`):**
The `y` dataframe contains a single column named `status`. This is the variable the model aims to predict. The `status` column is binary:
*   `0`: Indicates a healthy individual (absence of Parkinson's disease).
*   `1`: Indicates an individual with Parkinson's disease.

**What the k-Nearest Neighbors (KNN) Algorithm is Predicting:**
In this context, the KNN algorithm is a classifier that tries to predict whether an individual has Parkinson's disease (`status = 1`) or not (`status = 0`) based on their voice measurements (features in `X`).

When `model.predict(X_test)` is called, the algorithm looks at the `k` nearest data points (in this case, `k=3` as set in `KNeighborsClassifier(n_neighbors=3)`) in the training data (`X_train`) to each new data point in `X_test`. It then assigns the `status` label that is most common among these `k` nearest neighbors to the new data point. Essentially, it classifies an individual based on the `status` of similar individuals whose voice characteristics it has already learned from.

Install repository

In [1]:
pip install ucimlrepo

Imports for steps 1-6

In [2]:
from ucimlrepo import fetch_ucirepo # step 1
import pandas as pd
from sklearn.model_selection import train_test_split # step 2
from sklearn.neighbors import KNeighborsClassifier # step 3
from sklearn.metrics import accuracy_score # step 4
from sklearn.metrics import confusion_matrix, classification_report # step 5
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score # step 6

Step 1: Load the dataset and inspect it

In [3]:
# fetch dataset
parkinsons = fetch_ucirepo(id=174)

# data (as pandas dataframes)
X = parkinsons.data.features
y = parkinsons.data.targets

print(X.head())
print(y.head())

   MDVP:Fo  MDVP:Fhi  MDVP:Flo  MDVP:Jitter  MDVP:Jitter  MDVP:RAP  MDVP:PPQ  \
0  119.992   157.302    74.997      0.00784      0.00784   0.00370   0.00554   
1  122.400   148.650   113.819      0.00968      0.00968   0.00465   0.00696   
2  116.682   131.111   111.555      0.01050      0.01050   0.00544   0.00781   
3  116.676   137.871   111.366      0.00997      0.00997   0.00502   0.00698   
4  116.014   141.781   110.655      0.01284      0.01284   0.00655   0.00908   

   Jitter:DDP  MDVP:Shimmer  MDVP:Shimmer  ...  MDVP:APQ  Shimmer:DDA  \
0     0.01109       0.04374       0.04374  ...   0.02971      0.06545   
1     0.01394       0.06134       0.06134  ...   0.04368      0.09403   
2     0.01633       0.05233       0.05233  ...   0.03590      0.08270   
3     0.01505       0.05492       0.05492  ...   0.03772      0.08771   
4     0.01966       0.06425       0.06425  ...   0.04465      0.10470   

       NHR     HNR      RPDE       DFA   spread1   spread2        D2       PPE  

Step 2: split the data into training and testing sets

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Step 3: train the classifier

In [10]:
model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


KNeighborsClassifier(n_neighbors=3)

Step 4: validate the model

In [11]:
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")

Accuracy: 0.79


Step 5 evaluate the performance

In [12]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Predicted vs Actual")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.savefig('predicated-vs-actual.png')
plt.close()

Classification Report

In [13]:
print(classification_report(y_test, y_pred, target_names=parkinsons.target_names))

              precision    recall  f1-score   support

           0       0.43      0.43      0.43         7
           1       0.88      0.88      0.88        32

    accuracy                           0.79        39
   macro avg       0.65      0.65      0.65        39
weighted avg       0.79      0.79      0.79        39



Step 6 cross-validation

In [14]:
scores = cross_val_score(model, X, y, cv=5)
print(f"Cross-Validation Accuracy: {scores.mean():.2f}")

Cross-Validation Accuracy: 0.72


/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for 